In [6]:
import zipfile
import os
from pathlib import Path

# 1. Unzip the dataset uploaded in Colab
zip_path = 'Student folder.zip'
raw_extract = Path('/tmp/raw_data')

if not raw_extract.exists():
    print("Extracting Student folder.zip...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(raw_extract)
    print("Extraction complete!")

# 2. Create project directories
pkg_root = Path('/tmp/pathsnap_p1')
for d in ['src/data', 'data/processed', 'reports/figures']:
    (pkg_root / d).mkdir(parents=True, exist_ok=True)

print(f"Workspace ready at {pkg_root}")

Workspace ready at /tmp/pathsnap_p1


In [7]:
import os
from pathlib import Path

pkg_root = Path('/tmp/pathsnap_p1')

# Write loader.py
(pkg_root / 'src' / 'data' / 'loader.py').write_text('''from pathlib import Path
import numpy as np
import pandas as pd

M_PER_DEG_LAT = 111132.95

def load_trip(trip_dir: Path) -> dict:
    trip_dir = Path(trip_dir)
    tid = trip_dir.name
    track = pd.read_csv(trip_dir / f"{tid}.track", header=None, sep=r"\\s+", names=["longitude", "latitude", "timestamp"])
    nodes = pd.read_csv(trip_dir / f"{tid}.nodes", header=None, sep=r"\\s+", names=["longitude", "latitude"])
    arcs = pd.read_csv(trip_dir / f"{tid}.arcs", header=None, sep=r"\\s+", names=["from_node_id", "to_node_id"])
    arcs["arc_id"] = np.arange(len(arcs), dtype=np.int32)
    route_file = trip_dir / f"{tid}.route"
    route = None
    if route_file.exists():
        route = pd.read_csv(route_file, header=None, sep=r"\\s+", names=["arc_id"])["arc_id"].to_numpy(dtype=np.int32)
    return {"trip_id": tid, "track": track, "nodes": nodes, "arcs": arcs, "route": route}

def latlon_to_local_xy(lon: np.ndarray, lat: np.ndarray, lat0: float, lon0: float):
    m_per_deg_lon = 111319.49 * np.cos(np.radians(lat0))
    x = (lon - lon0) * m_per_deg_lon
    y = (lat - lat0) * M_PER_DEG_LAT
    return x, y
''')

# Write label_alignment.py
(pkg_root / 'src' / 'data' / 'label_alignment.py').write_text('''import numpy as np
from .loader import latlon_to_local_xy

def point_to_segments_dist_m(px, py, ax, ay, bx, by):
    dx = bx - ax
    dy = by - ay
    len_sq = dx * dx + dy * dy
    safe_len_sq = np.where(len_sq < 1e-9, 1.0, len_sq)
    t = ((px[:, None] - ax[None, :]) * dx[None, :] + (py[:, None] - ay[None, :]) * dy[None, :]) / safe_len_sq[None, :]
    t = np.clip(t, 0.0, 1.0)
    proj_x = ax[None, :] + t * dx[None, :]
    proj_y = ay[None, :] + t * dy
    dist = np.hypot(px[:, None] - proj_x, py[:, None] - proj_y)
    return dist, t

def align_track_to_route(track_df, nodes_df, arcs_df, route_arcs: np.ndarray) -> np.ndarray:
    lat0 = float(track_df["latitude"].mean())
    lon0 = float(track_df["longitude"].mean())
    px, py = latlon_to_local_xy(track_df["longitude"].to_numpy(), track_df["latitude"].to_numpy(), lat0, lon0)
    nx, ny = latlon_to_local_xy(nodes_df["longitude"].to_numpy(), nodes_df["latitude"].to_numpy(), lat0, lon0)
    from_ids = arcs_df["from_node_id"].to_numpy()[route_arcs]
    to_ids = arcs_df["to_node_id"].to_numpy()[route_arcs]
    ax, ay = nx[from_ids], ny[from_ids]
    bx, by = nx[to_ids], ny[to_ids]
    dist_mat, _ = point_to_segments_dist_m(px, py, ax, ay, bx, by)
    N, M = dist_mat.shape
    dp = np.full((N, M), np.inf, dtype=np.float64)
    back = np.zeros((N, M), dtype=np.int32)
    dp[0] = dist_mat[0]
    for i in range(1, N):
        prefix_min = np.minimum.accumulate(dp[i - 1])
        best_k = np.zeros(M, dtype=np.int32)
        cur_idx = 0
        for j in range(1, M):
            if dp[i - 1, j] < dp[i - 1, cur_idx]:
                cur_idx = j
            best_k[j] = cur_idx
        dp[i] = dist_mat[i] + prefix_min + 0.5 * (np.arange(M) - best_k)
        back[i] = best_k
    seq_idx = np.zeros(N, dtype=np.int32)
    seq_idx[-1] = int(np.argmin(dp[-1]))
    for i in range(N - 2, -1, -1):
        seq_idx[i] = back[i + 1, seq_idx[i + 1]]
    return route_arcs[seq_idx]
''')

# Write candidates.py
(pkg_root / 'src' / 'data' / 'candidates.py').write_text('''import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from .loader import latlon_to_local_xy
from .label_alignment import align_track_to_route

def build_trip_candidates(trip_data: dict, radius_m: float = 75.0, max_candidates: int = 15, include_labels: bool = True):
    tid = trip_data["trip_id"]
    track = trip_data["track"].copy()
    nodes = trip_data["nodes"]
    arcs = trip_data["arcs"]
    route = trip_data["route"]
    lat0 = float(track["latitude"].mean())
    lon0 = float(track["longitude"].mean())
    px, py = latlon_to_local_xy(track["longitude"].to_numpy(), track["latitude"].to_numpy(), lat0, lon0)
    nx, ny = latlon_to_local_xy(nodes["longitude"].to_numpy(), nodes["latitude"].to_numpy(), lat0, lon0)
    from_ids = arcs["from_node_id"].to_numpy()
    to_ids = arcs["to_node_id"].to_numpy()
    ax, ay = nx[from_ids], ny[from_ids]
    bx, by = nx[to_ids], ny[to_ids]
    arc_dx = bx - ax
    arc_dy = by - ay
    arc_len = np.hypot(arc_dx, arc_dy)
    arc_heading = np.arctan2(arc_dy, arc_dx)
    mid_x = 0.5 * (ax + bx)
    mid_y = 0.5 * (ay + by)
    n_nodes = len(nodes)
    out_deg = np.bincount(from_ids, minlength=n_nodes)
    in_deg = np.bincount(to_ids, minlength=n_nodes)
    arc_from_out_deg = out_deg[from_ids]
    arc_to_out_deg = out_deg[to_ids]
    arc_from_in_deg = in_deg[from_ids]
    ts = track["timestamp"].to_numpy(dtype=np.float64)
    dt = np.diff(ts, prepend=ts[0])
    dt[0] = 1.0
    dt = np.where(dt <= 0, 1.0, dt)
    step_dx = np.diff(px, prepend=px[0])
    step_dy = np.diff(py, prepend=py[0])
    if len(px) > 1:
        step_dx[0] = px[1] - px[0]
        step_dy[0] = py[1] - py[0]
    step_dist = np.hypot(step_dx, step_dy)
    speed_mps = step_dist / dt
    accel_mps2 = np.diff(speed_mps, prepend=speed_mps[0]) / dt
    gps_heading = np.arctan2(step_dy, step_dx)
    true_arc_per_obs = None
    if include_labels and route is not None:
        true_arc_per_obs = align_track_to_route(track, nodes, arcs, route)
    max_half_len = float(np.percentile(arc_len, 99) * 0.5) + 50.0
    tree = cKDTree(np.column_stack([mid_x, mid_y]))
    nearby_lists = tree.query_ball_point(np.column_stack([px, py]), r=radius_m + max_half_len)
    rows = []
    missed_obs = 0
    total_obs = len(track)
    for i in range(total_obs):
        cand_ids = np.asarray(nearby_lists[i], dtype=np.int32)
        if len(cand_ids) == 0:
            _, cand_ids = tree.query([px[i], py[i]], k=min(5, len(arcs)))
            cand_ids = np.atleast_1d(cand_ids).astype(np.int32)
        c_ax, c_ay = ax[cand_ids], ay[cand_ids]
        c_dx, c_dy = arc_dx[cand_ids], arc_dy[cand_ids]
        c_len_sq = np.where(arc_len[cand_ids] ** 2 < 1e-9, 1.0, arc_len[cand_ids] ** 2)
        t = ((px[i] - c_ax) * c_dx + (py[i] - c_ay) * c_dy) / c_len_sq
        t_clamped = np.clip(t, 0.0, 1.0)
        proj_x = c_ax + t_clamped * c_dx
        proj_y = c_ay + t_clamped * c_dy
        perp_dist = np.hypot(px[i] - proj_x, py[i] - proj_y)
        keep_mask = perp_dist <= radius_m
        if not np.any(keep_mask):
            order = np.argsort(perp_dist)[:max_candidates]
        else:
            valid_idx = np.where(keep_mask)[0]
            order = valid_idx[np.argsort(perp_dist[valid_idx])[:max_candidates]]
        sel_arcs = cand_ids[order]
        sel_dist = perp_dist[order]
        sel_t = t_clamped[order]
        if true_arc_per_obs is not None:
            if true_arc_per_obs[i] not in sel_arcs:
                missed_obs += 1
        angle_diff = np.abs(np.arctan2(np.sin(gps_heading[i] - arc_heading[sel_arcs]), np.cos(gps_heading[i] - arc_heading[sel_arcs])))
        cos_heading_diff = np.cos(angle_diff)
        sin_heading_diff = np.sin(angle_diff)
        min_d = sel_dist.min()
        dist_rank = np.arange(len(sel_arcs), dtype=np.float32)
        dist_diff_from_best = sel_dist - min_d
        dist_ratio_to_best = (sel_dist + 0.1) / (min_d + 0.1)
        gaussian_emission = np.exp(-0.5 * (sel_dist / 10.0) ** 2)
        for k in range(len(sel_arcs)):
            aid = int(sel_arcs[k])
            row = {"trip_id": tid, "obs_idx": int(i), "arc_id": aid}
            if true_arc_per_obs is not None:
                row["label"] = int(aid == true_arc_per_obs[i])
            row.update({
                "perp_dist_m": float(sel_dist[k]), "proj_ratio_t": float(sel_t[k]),
                "arc_length_m": float(arc_len[aid]), "gaussian_emission": float(gaussian_emission[k]),
                "heading_diff_rad": float(angle_diff[k]), "cos_heading_diff": float(cos_heading_diff[k]),
                "sin_heading_diff": float(sin_heading_diff[k]), "dt_sec": float(dt[i]),
                "step_dist_m": float(step_dist[i]), "speed_mps": float(speed_mps[i]),
                "accel_mps2": float(accel_mps2[i]), "is_stationary": int(speed_mps[i] < 0.5),
                "from_out_deg": int(arc_from_out_deg[aid]), "to_out_deg": int(arc_to_out_deg[aid]),
                "from_in_deg": int(arc_from_in_deg[aid]), "dist_rank": float(dist_rank[k]),
                "dist_diff_from_best_m": float(dist_diff_from_best[k]), "dist_ratio_to_best": float(dist_ratio_to_best[k]),
                "num_candidates": int(len(sel_arcs)),
            })
            rows.append(row)
    stats = {"trip_id": tid, "total_obs": total_obs, "missed_obs": missed_obs, "candidate_miss_rate": missed_obs / max(total_obs, 1), "avg_candidates_per_obs": len(rows) / max(total_obs, 1)}
    return pd.DataFrame(rows), stats
''')

# Write preprocess.py
(pkg_root / 'src' / 'data' / 'preprocess.py').write_text('''import joblib
import pandas as pd
from sklearn.preprocessing import RobustScaler

FEATURE_COLS = [
    "perp_dist_m", "proj_ratio_t", "arc_length_m", "gaussian_emission",
    "heading_diff_rad", "cos_heading_diff", "sin_heading_diff", "dt_sec",
    "step_dist_m", "speed_mps", "accel_mps2", "is_stationary",
    "from_out_deg", "to_out_deg", "from_in_deg", "dist_rank",
    "dist_diff_from_best_m", "dist_ratio_to_best", "num_candidates",
]

def fit_and_transform_splits(train_df: pd.DataFrame, student_df: pd.DataFrame, artifact_path: str):
    scaler = RobustScaler()
    train_scaled = train_df.copy()
    student_scaled = student_df.copy()
    scaled_cols = [f"{c}_scaled" for c in FEATURE_COLS]
    train_scaled[scaled_cols] = scaler.fit_transform(train_df[FEATURE_COLS])
    student_scaled[scaled_cols] = scaler.transform(student_df[FEATURE_COLS])
    joblib.dump({"scaler": scaler, "feature_cols": FEATURE_COLS, "scaled_cols": scaled_cols}, artifact_path)
    return train_scaled, student_scaled
''')

print("All module files successfully written to /tmp/pathsnap_p1/src/data/")

All module files successfully written to /tmp/pathsnap_p1/src/data/


In [8]:
import sys
from pathlib import Path
import pandas as pd

# Add path so python can load our modules
sys.path.insert(0, '/tmp/pathsnap_p1')
from src.data.loader import load_trip
from src.data.candidates import build_trip_candidates
from src.data.preprocess import fit_and_transform_splits

raw_root = Path('/tmp/raw_data/Student folder')
out_dir = Path('data/processed')
out_dir.mkdir(parents=True, exist_ok=True)

print("Processing 80 training trips...")
train_dfs, train_stats = [], []
for trip_dir in sorted((raw_root / 'train').iterdir()):
    if trip_dir.is_dir():
        data = load_trip(trip_dir)
        df_cand, st = build_trip_candidates(data, radius_m=75.0, max_candidates=15, include_labels=True)
        train_dfs.append(df_cand)
        train_stats.append(st)

train_all = pd.concat(train_dfs, ignore_index=True)
train_stats_df = pd.DataFrame(train_stats)

print("Processing 10 student-test trips...")
student_dfs, student_stats = [], []
for trip_dir in sorted((raw_root / 'student_test').iterdir()):
    if trip_dir.is_dir():
        data = load_trip(trip_dir)
        df_cand, st = build_trip_candidates(data, radius_m=75.0, max_candidates=15, include_labels=True)
        student_dfs.append(df_cand)
        student_stats.append(st)

student_all = pd.concat(student_dfs, ignore_index=True)
student_stats_df = pd.DataFrame(student_stats)

# Fit scaler strictly on train and save parquet datasets
train_final, student_final = fit_and_transform_splits(
    train_all, student_all, str(out_dir / 'preprocessor.joblib')
)

train_final.to_parquet(out_dir / 'candidates_train.parquet', index=False)
student_final.to_parquet(out_dir / 'candidates_student.parquet', index=False)

print("\n=== PIPELINE EXECUTION SUCCESSFUL ===")
print(f"Saved: data/processed/candidates_train.parquet ({train_final.shape})")
print(f"Saved: data/processed/candidates_student.parquet ({student_final.shape})")
print(f"Train Candidate Miss Rate: {train_stats_df['candidate_miss_rate'].mean():.4%}")
print(f"Student Candidate Miss Rate: {student_stats_df['candidate_miss_rate'].mean():.4%}")

Processing 80 training trips...
Processing 10 student-test trips...

=== PIPELINE EXECUTION SUCCESSFUL ===
Saved: data/processed/candidates_train.parquet ((2065006, 42))
Saved: data/processed/candidates_student.parquet ((203611, 42))
Train Candidate Miss Rate: 3.3254%
Student Candidate Miss Rate: 1.6951%


In [9]:
import pandas as pd

train_df = pd.read_parquet('data/processed/candidates_train.parquet')
print("--- TRAIN PARQUET SUMMARY ---")
print("Columns:", list(train_df.columns[:10]), "...", list(train_df.columns[-5:]))
print("Total rows:", len(train_df))
print("Unique trips:", train_df['trip_id'].nunique())
print("Label distribution:\n", train_df['label'].value_counts(normalize=True))

student_df = pd.read_parquet('data/processed/candidates_student.parquet')
print("\n--- STUDENT TEST PARQUET SUMMARY ---")
print("Total rows:", len(student_df))
print("Unique trips:", student_df['trip_id'].nunique())

--- TRAIN PARQUET SUMMARY ---
Columns: ['trip_id', 'obs_idx', 'arc_id', 'label', 'perp_dist_m', 'proj_ratio_t', 'arc_length_m', 'gaussian_emission', 'heading_diff_rad', 'cos_heading_diff'] ... ['from_in_deg_scaled', 'dist_rank_scaled', 'dist_diff_from_best_m_scaled', 'dist_ratio_to_best_scaled', 'num_candidates_scaled']
Total rows: 2065006
Unique trips: 80
Label distribution:
 label
0    0.908135
1    0.091865
Name: proportion, dtype: float64

--- STUDENT TEST PARQUET SUMMARY ---
Total rows: 203611
Unique trips: 10


In [11]:
import sys, json, shutil, zipfile
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, '/tmp/pathsnap_p1')
from src.data.loader import load_trip, latlon_to_local_xy
from src.data.label_alignment import align_track_to_route
from src.data.preprocess import FEATURE_COLS

raw_root = Path('/tmp/raw_data/Student folder')
out_dir  = Path('data/processed'); out_dir.mkdir(parents=True, exist_ok=True)
SPLITS = {'train': 'train', 'student': 'student_test'}   # name -> raw folder

# ---------- 1. validation of candidate tables ----------
cand = {s: pd.read_parquet(out_dir / f'candidates_{s}.parquet') for s in SPLITS}
assert not (set(cand['train'].trip_id) & set(cand['student'].trip_id)), "trip leakage!"
for s, df in cand.items():
    assert df.groupby('trip_id')['obs_idx'].apply(lambda x: x.is_monotonic_increasing).all(), f"{s}: obs_idx not in time order"
    assert df.groupby(['trip_id', 'obs_idx'])['label'].sum().max() <= 1, f"{s}: >1 positive per obs"
    assert not df[FEATURE_COLS].isna().any().any(), f"{s}: NaNs in features"
    print(f"{s}: shape={df.shape}, trips={df.trip_id.nunique()}, pos rate={df.label.mean():.4f}")

# ---------- 2. observations + per-trip graph exports ----------
miss_rows = []
for s, folder in SPLITS.items():
    obs_l, arcs_l, nodes_l, route_l, meta_l = [], [], [], [], []
    cdf = cand[s]
    hit = cdf.groupby(['trip_id', 'obs_idx'])['label'].max()
    ncand = cdf.groupby(['trip_id', 'obs_idx']).size()
    for trip_dir in sorted((raw_root / folder).iterdir()):
        if not trip_dir.is_dir(): continue
        d = load_trip(trip_dir); tid = d['trip_id']
        trk, nodes, arcs, route = d['track'], d['nodes'], d['arcs'], d['route']
        lat0, lon0 = float(trk.latitude.mean()), float(trk.longitude.mean())
        px, py = latlon_to_local_xy(trk.longitude.values, trk.latitude.values, lat0, lon0)
        nx, ny = latlon_to_local_xy(nodes.longitude.values, nodes.latitude.values, lat0, lon0)
        ts = trk.timestamp.values.astype(float)
        dt = np.diff(ts, prepend=ts[0]); dt[0] = 1.0; dt = np.where(dt <= 0, 1.0, dt)
        sdx = np.diff(px, prepend=px[0]); sdy = np.diff(py, prepend=py[0])
        if len(px) > 1: sdx[0], sdy[0] = px[1]-px[0], py[1]-py[0]
        step = np.hypot(sdx, sdy)
        ref = align_track_to_route(trk, nodes, arcs, route) if route is not None else np.full(len(trk), -1)
        idx = pd.MultiIndex.from_arrays([[tid]*len(trk), np.arange(len(trk))])
        obs_l.append(pd.DataFrame({
            'trip_id': tid, 'obs_idx': np.arange(len(trk)),
            'longitude': trk.longitude.values, 'latitude': trk.latitude.values,
            'timestamp': ts, 'x_m': px, 'y_m': py, 'dt_sec': dt,
            'step_dist_m': step, 'speed_mps': step/dt, 'gps_heading_rad': np.arctan2(sdy, sdx),
            'ref_arc_id': ref.astype(int)
        }))

train: shape=(2065006, 42), trips=80, pos rate=0.0919
student: shape=(203611, 42), trips=10, pos rate=0.0920


In [14]:
from pathlib import Path
import os, shutil
from google.colab import drive

drive.mount('/content/drive')

src  = Path('data/processed')
dest = Path('/content/drive/MyDrive/ML HACKATHON')   # your existing folder
dest.mkdir(parents=True, exist_ok=True)

assert src.exists() and (src / 'candidates_train.parquet').exists(), \
    "Build files missing in data/processed - run the earlier big cell first (check it finished without errors)."

# Free space on Drive
tot, used, free = shutil.disk_usage('/content/drive')
print(f"Drive free: {free/1e9:.2f} GB")

# 1. Small/medium files first (skip ones already in the folder)
to_copy = [fp for fp in sorted(src.glob('*')) if fp.is_file() and not (dest / fp.name).exists()]
need = sum(fp.stat().st_size for fp in to_copy)
print(f"New files to upload: {len(to_copy)}  ({need/1e6:.1f} MB)")
assert need < free * 0.9, "Not enough Drive space - free some space or tell me and I'll slim the files."

for fp in to_copy:
    shutil.copy(fp, dest / fp.name)
    print("copied", fp.name)

# 2. Raw per-trip files (.arcs/.nodes/.route/.track) as one zip, only if it fits
raw_zip = dest / 'raw_trips.zip'
if not raw_zip.exists() and (src / 'raw_trips').exists():
    tmp = shutil.make_archive('/content/raw_trips', 'zip', src / 'raw_trips')
    sz = os.path.getsize(tmp)
    _, _, free = shutil.disk_usage('/content/drive')
    if sz < free * 0.9:
        shutil.copy(tmp, raw_zip); print(f"copied raw_trips.zip ({sz/1e6:.1f} MB)")
    else:
        print(f"raw_trips.zip is {sz/1e6:.1f} MB - skipped, not enough space")

# 3. Final listing
print("\nContents of ML HACKATHON:")
for fp in sorted(dest.glob('*')):
    print(f"{fp.name:40s} {fp.stat().st_size/1e6:8.1f} MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive free: 2.49 GB
New files to upload: 3  (256.5 MB)
copied candidates_student.parquet
copied candidates_train.parquet
copied preprocessor.joblib

Contents of ML HACKATHON:
candidates_student.parquet                   25.4 MB
candidates_train.parquet                    231.0 MB
preprocessor.joblib                           0.0 MB


In [15]:
from pathlib import Path
SRC = Path('/tmp/pathsnap_p1/src'); SRC.mkdir(parents=True, exist_ok=True)
(SRC / 'pathsnap.py').write_text(r'''"""PathSnap shared helpers, schema 1.0. All coordinates are (lon, lat) WGS84 degrees."""
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

SCHEMA_VERSION = "1.0"
RADIUS_M = 75.0          # candidate search radius (point-to-segment distance)
MAX_CANDIDATES = 20      # max candidates kept per GPS point (closest first)
GAP_S = 30.0             # dt_prev > GAP_S  -> is_gap
LONG_ARC_M = 500.0       # arcs longer than this are checked against every point directly
AMBIG_M = 5.0            # another route arc within 5 m of the assigned arc -> 'ambiguous'
FAR_M = 50.0             # assigned arc farther than this -> 'far'
ALIGN_LAMBDA = 0.5       # metres of penalty per route position skipped forward
RECALL_RADII = (20, 30, 50, 75, 100)
M_PER_DEG_LAT = 111132.95
M_PER_DEG_LON_EQ = 111319.49
PROJECTION = ("Local equirectangular tangent plane in metres: x = east, y = north, origin = (lon0, lat0) "
              "= mean of the cleaned GPS track of that trip (stored in trips_meta_*.csv). "
              "Input degrees are assumed WGS84 lon/lat.")

OBS_COLS_LABELED = ["trip_id", "obs_idx", "orig_row", "timestamp", "dt_prev", "is_gap", "lon", "lat", "x", "y",
                    "speed", "heading", "n_candidates", "true_arc_id", "label_quality", "heading_valid", "true_dist_m"]
CAND_COLS = ["trip_id", "obs_idx", "arc_id", "from_node", "to_node", "dist_to_arc_m", "snap_x", "snap_y",
             "offset_m", "arc_length_m", "bearing_diff_deg", "cand_rank", "label"]

# ---------------- geometry helpers ----------------
def project(lon, lat, lat0, lon0):
    lon = np.asarray(lon, dtype=float); lat = np.asarray(lat, dtype=float)
    return ((lon - lon0) * M_PER_DEG_LON_EQ * np.cos(np.radians(lat0)), (lat - lat0) * M_PER_DEG_LAT)

def unproject(x, y, lat0, lon0):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    return (lon0 + x / (M_PER_DEG_LON_EQ * np.cos(np.radians(lat0))), lat0 + y / M_PER_DEG_LAT)

def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371008.8
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2 - p1
    dl = np.radians(np.asarray(lon2, dtype=float) - np.asarray(lon1, dtype=float))
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def point_segment_dist(px, py, ax, ay, bx, by):
    """Distance (m) from every point (N) to every segment (M) -> (N, M)."""
    dx, dy = bx - ax, by - ay
    l2 = dx * dx + dy * dy
    l2s = np.where(l2 < 1e-9, 1.0, l2)
    t = np.clip(((px[:, None] - ax) * dx + (py[:, None] - ay) * dy) / l2s, 0.0, 1.0)
    return np.hypot(px[:, None] - (ax + t * dx), py[:, None] - (ay + t * dy))

def _bearing_deg(dx, dy):
    """Compass bearing, degrees clockwise from north, in [0, 360)."""
    return (90.0 - np.degrees(np.arctan2(dy, dx))) % 360.0

# ---------------- loading ----------------
def _read(path, names):
    return pd.read_csv(path, header=None, sep=r"\s+", names=names, usecols=range(len(names)))

def find_trip_dir(trip_id, raw_root):
    raw_root = Path(raw_root)
    for sub in sorted(p for p in raw_root.iterdir() if p.is_dir()):
        if (sub / trip_id).is_dir():
            return sub / trip_id
    if (raw_root / trip_id).is_dir():
        return raw_root / trip_id
    raise FileNotFoundError(trip_id)

def load_trip(trip_id, raw_root=None, data_dir=None):
    """Return {'trip_id','track','nodes','arcs','route'}.
    raw_root: folder containing train/ student_test/ ... (raw .track/.nodes/.arcs/.route files)
    data_dir: folder with the delivered parquets (works without the raw zip)
    track: longitude, latitude, timestamp (+orig_row in parquet mode); nodes: longitude, latitude;
    arcs: from_node_id, to_node_id, arc_id (= row index in .arcs, PER TRIP); route: int array of arc_id or None."""
    if raw_root is not None:
        d = find_trip_dir(trip_id, raw_root)
        track = _read(d / f"{trip_id}.track", ["longitude", "latitude", "timestamp"])
        nodes = _read(d / f"{trip_id}.nodes", ["longitude", "latitude"])
        arcs = _read(d / f"{trip_id}.arcs", ["from_node_id", "to_node_id"])
        arcs["arc_id"] = np.arange(len(arcs), dtype=np.int32)
        rf = d / f"{trip_id}.route"
        route = _read(rf, ["arc_id"])["arc_id"].to_numpy(np.int64) if rf.exists() and rf.stat().st_size > 0 else None
        return {"trip_id": trip_id, "track": track, "nodes": nodes, "arcs": arcs, "route": route}
    if data_dir is not None:
        data_dir = Path(data_dir)
        for split in ("train", "student"):
            fo = data_dir / f"observations_{split}.parquet"
            if not fo.exists():
                continue
            flt = [("trip_id", "==", trip_id)]
            obs = pd.read_parquet(fo, filters=flt)
            if len(obs) == 0:
                continue
            arcs = pd.read_parquet(data_dir / f"arcs_{split}.parquet", filters=flt)
            nodes = pd.read_parquet(data_dir / f"nodes_{split}.parquet", filters=flt)
            rt = pd.read_parquet(data_dir / f"routes_{split}.parquet", filters=flt).sort_values("route_pos")
            track = obs[["orig_row", "lon", "lat", "timestamp"]].rename(columns={"lon": "longitude", "lat": "latitude"})
            return {"trip_id": trip_id, "track": track.reset_index(drop=True),
                    "nodes": nodes.rename(columns={"lon": "longitude", "lat": "latitude"})[["node_id", "longitude", "latitude"]].reset_index(drop=True),
                    "arcs": arcs.rename(columns={"from_node": "from_node_id", "to_node": "to_node_id"})[["arc_id", "from_node_id", "to_node_id"]].reset_index(drop=True),
                    "route": rt["arc_id"].to_numpy(np.int64) if len(rt) else None}
        raise FileNotFoundError(trip_id)
    raise ValueError("pass raw_root= or data_dir=")

# ---------------- cleaning ----------------
def clean_track(track):
    """Drops invalid coordinates, stable-sorts by timestamp if out of order. Duplicate timestamps are KEPT
    (dt_prev == 0). orig_row = 0-based row in the raw .track file."""
    t = track[["longitude", "latitude", "timestamp"]].copy().reset_index(drop=True)
    t["orig_row"] = np.arange(len(t), dtype=np.int64)
    for c in ("longitude", "latitude", "timestamp"):
        t[c] = pd.to_numeric(t[c], errors="coerce")
    bad = (t.isna().any(axis=1) | (t.longitude.abs() > 180) | (t.latitude.abs() > 90)
           | ((t.longitude == 0) & (t.latitude == 0)))
    log = {"n_raw_rows": int(len(t)), "n_invalid_coord": int(bad.sum())}
    t = t.loc[~bad].reset_index(drop=True)
    log["n_out_of_order"] = int((np.diff(t.timestamp.to_numpy(float)) < 0).sum())
    if log["n_out_of_order"]:
        t = t.sort_values("timestamp", kind="stable").reset_index(drop=True)
    log["n_dup_timestamp"] = int((np.diff(t.timestamp.to_numpy(float)) == 0).sum())
    log["n_dup_rows"] = int(t.duplicated(["longitude", "latitude", "timestamp"]).sum())
    return t, log

# ---------------- label-free build (API-safe) ----------------
def build_observations_and_candidates(trip_id, track, nodes, arcs, radius_m=RADIUS_M,
                                      max_candidates=MAX_CANDIDATES, gap_s=GAP_S):
    """LABEL-FREE. Needs only track, nodes, arcs. Returns (obs_df, cand_df, info)."""
    t, clog = clean_track(track)
    n = len(t)
    if n == 0:
        raise ValueError(f"{trip_id}: no valid GPS rows")
    lon = t.longitude.to_numpy(float); lat = t.latitude.to_numpy(float); ts = t.timestamp.to_numpy(float)
    lat0, lon0 = float(lat.mean()), float(lon.mean())
    px, py = project(lon, lat, lat0, lon0)
    nx, ny = project(nodes.longitude.to_numpy(float), nodes.latitude.to_numpy(float), lat0, lon0)
    fr = arcs.from_node_id.to_numpy(np.int64); to = arcs.to_node_id.to_numpy(np.int64)
    ax, ay, bx, by = nx[fr], ny[fr], nx[to], ny[to]
    dx, dy = bx - ax, by - ay
    alen = np.hypot(dx, dy)
    abear = _bearing_deg(dx, dy)

    dtp = np.diff(ts, prepend=ts[0]).astype(float)            # 0 for the first point
    is_gap = dtp > gap_s
    sdx = np.diff(px, prepend=px[0]); sdy = np.diff(py, prepend=py[0])
    step = np.hypot(sdx, sdy)
    speed = step / np.maximum(dtp, 1.0)                       # dt floored at 1 s (handles dt = 0)
    valid = step >= 1.0; valid[0] = False                     # heading only trusted if moved >= 1 m
    head = np.where(valid, _bearing_deg(sdx, sdy), np.nan)
    head = pd.Series(head).ffill().bfill().fillna(0.0).to_numpy()   # carry forward when stationary

    pts = np.column_stack([px, py])
    mid = np.column_stack([0.5 * (ax + bx), 0.5 * (ay + by)])
    is_long = alen > LONG_ARC_M
    short_ids = np.where(~is_long)[0]; long_ids = np.where(is_long)[0]
    if len(short_ids):
        tree = cKDTree(mid[short_ids])
        near = tree.query_ball_point(pts, r=radius_m + 0.5 * float(alen[short_ids].max()))
    else:
        near = [[] for _ in range(n)]

    C = {k: [] for k in ("obs_idx", "arc_id", "dist", "sx", "sy", "off", "bd", "rank")}
    ncand = np.zeros(n, dtype=np.int16); n_trunc = 0
    for i in range(n):
        ids = short_ids[np.asarray(near[i], dtype=np.int64)] if len(short_ids) else np.empty(0, np.int64)
        if long_ids.size:
            ids = np.concatenate([ids, long_ids])
        if ids.size == 0:
            continue
        l2 = alen[ids] ** 2
        tt = np.clip(((px[i] - ax[ids]) * dx[ids] + (py[i] - ay[ids]) * dy[ids]) / np.where(l2 < 1e-9, 1.0, l2), 0.0, 1.0)
        sx = ax[ids] + tt * dx[ids]; sy = ay[ids] + tt * dy[ids]
        d = np.hypot(px[i] - sx, py[i] - sy)
        m = d <= radius_m
        if not m.any():
            continue
        ids, tt, sx, sy, d = ids[m], tt[m], sx[m], sy[m], d[m]
        if len(ids) > max_candidates:
            n_trunc += 1
        o = np.lexsort((ids, d))[:max_candidates]           # closest first, ties by arc_id
        ids, tt, sx, sy, d = ids[o], tt[o], sx[o], sy[o], d[o]
        k = len(ids); ncand[i] = k
        C["obs_idx"].append(np.full(k, i, np.int32)); C["arc_id"].append(ids.astype(np.int32))
        C["dist"].append(d.astype(np.float32)); C["sx"].append(sx.astype(np.float32)); C["sy"].append(sy.astype(np.float32))
        C["off"].append((tt * alen[ids]).astype(np.float32))
        C["bd"].append((np.abs((head[i] - abear[ids] + 180.0) % 360.0 - 180.0)).astype(np.float32))
        C["rank"].append(np.arange(k, dtype=np.int16))

    cat = lambda k, dt: np.concatenate(C[k]) if C[k] else np.empty(0, dt)
    aid = cat("arc_id", np.int32)
    cand = pd.DataFrame({
        "trip_id": trip_id, "obs_idx": cat("obs_idx", np.int32), "arc_id": aid,
        "from_node": fr[aid].astype(np.int32), "to_node": to[aid].astype(np.int32),
        "dist_to_arc_m": cat("dist", np.float32), "snap_x": cat("sx", np.float32), "snap_y": cat("sy", np.float32),
        "offset_m": cat("off", np.float32), "arc_length_m": alen[aid].astype(np.float32),
        "bearing_diff_deg": cat("bd", np.float32), "cand_rank": cat("rank", np.int16)})
    obs = pd.DataFrame({
        "trip_id": trip_id, "obs_idx": np.arange(n, dtype=np.int32), "orig_row": t["orig_row"].to_numpy(np.int32),
        "timestamp": ts, "dt_prev": dtp.astype(np.float32), "is_gap": is_gap, "lon": lon, "lat": lat,
        "x": px, "y": py, "speed": speed.astype(np.float32), "heading": head.astype(np.float32),
        "n_candidates": ncand, "heading_valid": valid})
    info = {"trip_id": trip_id, "lat0": lat0, "lon0": lon0, "n_obs": n, "cleaning": clog,
            "n_zero_cand": int((ncand == 0).sum()), "n_truncated": int(n_trunc),
            "radius_m": radius_m, "max_candidates": max_candidates}
    return obs, cand, info

# ---------------- labels (train/student/server-side only) ----------------
def _monotone_align(D, lam=ALIGN_LAMBDA):
    """Viterbi over route positions: position never decreases; moving forward by k positions costs lam*k.
    D: (N points, M route arcs) distances. Returns route position per point."""
    N, M = D.shape
    idx = np.arange(M)
    dp = D[0].copy()
    back = np.zeros((N, M), dtype=np.int32)
    for i in range(1, N):
        tmp = dp - lam * idx
        cm = np.minimum.accumulate(tmp)
        back[i] = np.maximum.accumulate(np.where(tmp <= cm, idx, 0))
        dp = D[i] + lam * idx + cm
    pos = np.empty(N, dtype=np.int64)
    pos[-1] = int(np.argmin(dp))
    for i in range(N - 1, 0, -1):
        pos[i - 1] = back[i, pos[i]]
    return pos

def attach_labels(obs, cand, nodes, arcs, route, info, radius_m=RADIUS_M, recall_radii=RECALL_RADII):
    """Adds true_arc_id, label_quality, true_dist_m to obs and label to cand. Returns (obs, cand, lab_info)."""
    obs = obs.copy(); cand = cand.copy(); n = len(obs)
    empty = {"labeled": False, "n_labeled": 0, "d_true": np.empty(0), "n_hit": 0, "n_miss": 0,
             "n_miss_radius": 0, "n_miss_cap": 0, "quality": {}, "recall_counts": {r: 0 for r in recall_radii},
             "n_route_invalid": 0}
    if route is None or len(route) == 0 or n == 0:
        obs["true_arc_id"] = np.int32(-1); obs["label_quality"] = "unlabeled"; obs["true_dist_m"] = np.float32(np.nan)
        cand["label"] = np.int8(0)
        return obs[OBS_COLS_LABELED], cand[CAND_COLS], empty
    route = np.asarray(route, dtype=np.int64)
    ok = (route >= 0) & (route < len(arcs))
    ra = route[ok]
    nx, ny = project(nodes.longitude.to_numpy(float), nodes.latitude.to_numpy(float), info["lat0"], info["lon0"])
    fr = arcs.from_node_id.to_numpy(np.int64); to = arcs.to_node_id.to_numpy(np.int64)
    D = point_segment_dist(obs.x.to_numpy(float), obs.y.to_numpy(float), nx[fr[ra]], ny[fr[ra]], nx[to[ra]], ny[to[ra]])
    pos = _monotone_align(D)
    true_arc = ra[pos]
    ar = np.arange(n)
    d_true = D[ar, pos]
    D2 = D.copy(); D2[ra[None, :] == true_arc[:, None]] = np.inf
    d_alt = D2.min(axis=1)
    quality = np.select([d_true > FAR_M, (d_alt - d_true) < AMBIG_M, obs.is_gap.to_numpy()],
                        ["far", "ambiguous", "gap"], default="ok")
    obs["true_arc_id"] = true_arc.astype(np.int32)
    obs["label_quality"] = quality
    obs["true_dist_m"] = d_true.astype(np.float32)
    cand["label"] = (cand.arc_id.to_numpy() == true_arc[cand.obs_idx.to_numpy()]).astype(np.int8)
    hit = np.zeros(n, bool)
    hit[cand.obs_idx.to_numpy()[cand.label.to_numpy() == 1]] = True
    miss = ~hit; miss_r = miss & (d_true > radius_m); miss_c = miss & ~miss_r
    lab = {"labeled": True, "n_labeled": n, "d_true": d_true, "n_hit": int(hit.sum()), "n_miss": int(miss.sum()),
           "n_miss_radius": int(miss_r.sum()), "n_miss_cap": int(miss_c.sum()),
           "quality": {q: int((quality == q).sum()) for q in ("ok", "ambiguous", "far", "gap")},
           "recall_counts": {r: int((d_true <= r).sum()) for r in recall_radii},
           "n_route_invalid": int((~ok).sum())}
    return obs[OBS_COLS_LABELED], cand[CAND_COLS], lab

# ---------------- per-trip graph tables / audits ----------------
def graph_tables(trip_id, nodes, arcs, route, lat0, lon0):
    nlon = nodes.longitude.to_numpy(float); nlat = nodes.latitude.to_numpy(float)
    nx, ny = project(nlon, nlat, lat0, lon0)
    f = arcs.from_node_id.to_numpy(np.int64); t = arcs.to_node_id.to_numpy(np.int64)
    a = pd.DataFrame({"trip_id": trip_id, "arc_id": arcs.arc_id.to_numpy(np.int32),
                      "from_node": f.astype(np.int32), "to_node": t.astype(np.int32),
                      "from_lon": nlon[f], "from_lat": nlat[f], "to_lon": nlon[t], "to_lat": nlat[t],
                      "length_m": np.hypot(nx[t] - nx[f], ny[t] - ny[f]).astype(np.float32)})
    nd = pd.DataFrame({"trip_id": trip_id, "node_id": np.arange(len(nodes), dtype=np.int32),
                       "lon": nlon, "lat": nlat, "x": nx, "y": ny})
    rt = pd.DataFrame({"trip_id": trip_id, "route_pos": np.arange(0 if route is None else len(route), dtype=np.int32),
                       "arc_id": np.asarray([] if route is None else route, dtype=np.int32)})
    return a, nd, rt

def lonlat_order_check(track, nodes, pad=0.05):
    """Fraction of GPS points inside the node bounding box as (lon,lat) vs swapped (lat,lon)."""
    def inside(a, b, A, B):
        return float(((a >= A.min() - pad) & (a <= A.max() + pad) & (b >= B.min() - pad) & (b <= B.max() + pad)).mean())
    tl, tb = track.longitude.to_numpy(float), track.latitude.to_numpy(float)
    nl, nb = nodes.longitude.to_numpy(float), nodes.latitude.to_numpy(float)
    return inside(tl, tb, nl, nb), inside(tl, tb, nb, nl)

def graph_audit(nodes, arcs, route):
    n_nodes, n_arcs = len(nodes), len(arcs)
    f = arcs.from_node_id.to_numpy(np.int64); t = arcs.to_node_id.to_numpy(np.int64)
    out = {"n_nodes": n_nodes, "n_arcs": n_arcs,
           "arcs_bad_node_ref": int(((f < 0) | (f >= n_nodes) | (t < 0) | (t >= n_nodes)).sum())}
    fc, tc = np.clip(f, 0, n_nodes - 1), np.clip(t, 0, n_nodes - 1)
    lat0, lon0 = float(nodes.latitude.mean()), float(nodes.longitude.mean())
    nx, ny = project(nodes.longitude.to_numpy(float), nodes.latitude.to_numpy(float), lat0, lon0)
    length = np.hypot(nx[tc] - nx[fc], ny[tc] - ny[fc])
    out.update(self_loops=int((f == t).sum()), zero_len_arcs=int((length < 0.01).sum()),
               duplicate_arcs=int(pd.Series(fc * n_nodes + tc).duplicated().sum()),
               arc_len_p50=float(np.median(length)), arc_len_p99=float(np.percentile(length, 99)), arc_len_max=float(length.max()))
    key = fc * n_nodes + tc
    ns = f != t
    out["frac_arcs_with_reverse"] = float(np.isin(tc[ns] * n_nodes + fc[ns], key).mean()) if ns.any() else 0.0
    used = np.zeros(n_nodes, bool); used[fc] = True; used[tc] = True
    out["isolated_nodes"] = int((~used).sum())
    g = coo_matrix((np.ones(n_arcs), (fc, tc)), shape=(n_nodes, n_nodes))
    for nm, kw in (("weak", dict(directed=False)), ("strong", dict(directed=True, connection="strong"))):
        nc, lab = connected_components(g, **kw)
        sizes = np.bincount(lab[used], minlength=nc); sizes = sizes[sizes > 0]
        out[f"n_{nm}_components"] = int(len(sizes)); out[f"largest_{nm}_frac"] = float(sizes.max() / sizes.sum())
    z = dict(route_len=0, route_pairs=0, route_repeat=0, route_connected=0, route_uturn=0, route_disconnected=0, route_invalid_ids=0)
    if route is not None and len(route) > 1:
        r = np.asarray(route, dtype=np.int64); okr = (r >= 0) & (r < n_arcs)
        both = okr[:-1] & okr[1:]; a, b = r[:-1][both], r[1:][both]
        rep = a == b; con = (~rep) & (t[a] == f[b])
        ut = (~rep) & (~con) & (t[b] == f[a]) & (f[b] == t[a])
        z = dict(route_len=int(len(r)), route_pairs=int(len(a)), route_repeat=int(rep.sum()), route_connected=int(con.sum()),
                 route_uturn=int(ut.sum()), route_disconnected=int((~rep & ~con & ~ut).sum()), route_invalid_ids=int((~okr).sum()))
    out.update(z)
    return out

def route_continuity(arcs_tbl, arc_seq):
    """Fraction of consecutive predicted arcs that are the same arc or directly connected (to_node == next from_node).
    arcs_tbl: arcs_{split}.parquet rows of ONE trip (needs arc_id, from_node, to_node)."""
    tb = arcs_tbl.set_index("arc_id"); seq = list(arc_seq)
    ok = [(a == b) or (tb.at[a, "to_node"] == tb.at[b, "from_node"]) for a, b in zip(seq[:-1], seq[1:])]
    return sum(ok) / max(len(ok), 1)
''')
print("pathsnap.py written ->", SRC / 'pathsnap.py')

pathsnap.py written -> /tmp/pathsnap_p1/src/pathsnap.py


In [16]:
import sys, json, hashlib, importlib
import numpy as np, pandas as pd
from pathlib import Path
sys.path.insert(0, '/tmp/pathsnap_p1/src')
import pathsnap; importlib.reload(pathsnap)
from pathsnap import *

raw_root = Path('/tmp/raw_data/Student folder')
OUT = Path('/content/pathsnap_out'); OUT.mkdir(parents=True, exist_ok=True)

def split_key(name):
    n = name.lower()
    if 'student' in n: return 'student'
    if 'train' in n: return 'train'
    return 'evaluator'

splits = {'train': [], 'student': [], 'evaluator': []}; folders = {}
for folder in sorted(p for p in raw_root.iterdir() if p.is_dir()):
    tids = sorted(p.name for p in folder.iterdir() if p.is_dir())
    print(f"folder {folder.name!r}: {len(tids)} trips -> '{split_key(folder.name)}'")
    splits[split_key(folder.name)] += tids; folders[folder.name] = split_key(folder.name)
json.dump({'schema_version': SCHEMA_VERSION, **splits, 'raw_folders': folders,
           'note': 'evaluator is empty if no hidden/evaluator folder exists in the zip'},
          open(OUT / 'splits.json', 'w'), indent=2)

rows = []
for sp, tids in splits.items():
    for tid in tids:
        d = load_trip(tid, raw_root=raw_root); tdir = find_trip_dir(tid, raw_root)
        ct, clog = clean_track(d['track'])
        ts = ct.timestamp.to_numpy(float); gaps = np.diff(ts) if len(ts) > 1 else np.array([0.0])
        lg, sw = lonlat_order_check(ct, d['nodes'])
        lon, lat = ct.longitude.to_numpy(float), ct.latitude.to_numpy(float)
        px, py = project(lon, lat, lat.mean(), lon.mean())
        step = np.hypot(np.diff(px), np.diff(py)); hv = haversine_m(lon[:-1], lat[:-1], lon[1:], lat[1:])
        m = hv > 10
        rel = float(np.max(np.abs(step[m] - hv[m]) / hv[m])) if m.any() else 0.0
        g = graph_audit(d['nodes'], d['arcs'], d['route'])
        rows.append({'trip_id': tid, 'split': sp, **clog, 'n_obs_clean': len(ct),
                     'dt_gt1': int((gaps > 1).sum()), 'dt_gt5': int((gaps > 5).sum()), 'dt_gt30': int((gaps > 30).sum()),
                     'dt_max': float(gaps.max()), 'dt_median': float(np.median(gaps)),
                     'inside_bbox_lonlat': lg, 'inside_bbox_swapped': sw, 'proj_vs_haversine_max_rel_err': rel,
                     'nodes_md5': hashlib.md5((tdir / f'{tid}.nodes').read_bytes()).hexdigest(),
                     'arcs_md5': hashlib.md5((tdir / f'{tid}.arcs').read_bytes()).hexdigest(),
                     'has_route': d['route'] is not None, **g})
A = pd.DataFrame(rows); A.to_csv(OUT / 'audit_trips.csv', index=False)

lab = A[A.has_route]
S = {
 'n_trips': len(A), 'trips_per_split': A.split.value_counts().to_dict(),
 'lonlat_ok_trips': int(((A.inside_bbox_lonlat >= 0.95) & (A.inside_bbox_swapped < A.inside_bbox_lonlat)).sum()),
 'median_lon': float(A.groupby('trip_id').size().size and pd.concat([load_trip(t, raw_root=raw_root)['track'].longitude for t in A.trip_id[:3]]).median()),
 'median_lat': float(pd.concat([load_trip(t, raw_root=raw_root)['track'].latitude for t in A.trip_id[:3]]).median()),
 'proj_max_rel_err': float(A.proj_vs_haversine_max_rel_err.max()),
 'unique_nodes_files': int(A.nodes_md5.nunique()), 'unique_arcs_files': int(A.arcs_md5.nunique()),
 'frac_arcs_with_reverse_mean': float(A.frac_arcs_with_reverse.mean()),
 'route_pairs': int(lab.route_pairs.sum()), 'route_connected': int(lab.route_connected.sum()),
 'route_repeat': int(lab.route_repeat.sum()), 'route_uturn': int(lab.route_uturn.sum()),
 'route_disconnected': int(lab.route_disconnected.sum()), 'route_invalid_ids': int(lab.route_invalid_ids.sum()),
 'self_loops': int(A.self_loops.sum()), 'zero_len_arcs': int(A.zero_len_arcs.sum()), 'duplicate_arcs': int(A.duplicate_arcs.sum()),
 'isolated_nodes': int(A.isolated_nodes.sum()), 'arcs_bad_node_ref': int(A.arcs_bad_node_ref.sum()),
 'trips_multi_weak_components': int((A.n_weak_components > 1).sum()), 'trips_multi_strong_components': int((A.n_strong_components > 1).sum()),
 'min_largest_weak_frac': float(A.largest_weak_frac.min()), 'arc_len_max': float(A.arc_len_max.max()),
 'dt_gt1': int(A.dt_gt1.sum()), 'dt_gt5': int(A.dt_gt5.sum()), 'dt_gt30': int(A.dt_gt30.sum()), 'dt_max': float(A.dt_max.max()),
 'n_invalid_coord': int(A.n_invalid_coord.sum()), 'n_out_of_order': int(A.n_out_of_order.sum()),
 'n_dup_timestamp': int(A.n_dup_timestamp.sum()), 'n_dup_rows': int(A.n_dup_rows.sum()), 'n_raw_rows': int(A.n_raw_rows.sum()),
}
json.dump(S, open(OUT / 'audit_summary.json', 'w'), indent=2)
print(json.dumps(S, indent=2))
print("\nTrips with gaps > 30 s:", int((A.dt_gt30 > 0).sum()), "| largest arc (m):", round(S['arc_len_max'], 1))

folder 'student_test': 10 trips -> 'student'
folder 'train': 80 trips -> 'train'
{
  "n_trips": 90,
  "trips_per_split": {
    "train": 80,
    "student": 10
  },
  "lonlat_ok_trips": 90,
  "median_lon": 11.039095,
  "median_lat": 47.693349,
  "proj_max_rel_err": 0.008886478815836867,
  "unique_nodes_files": 90,
  "unique_arcs_files": 90,
  "frac_arcs_with_reverse_mean": 0.9222579437556072,
  "route_pairs": 34723,
  "route_connected": 34709,
  "route_repeat": 0,
  "route_uturn": 0,
  "route_disconnected": 14,
  "route_invalid_ids": 0,
  "self_loops": 24,
  "zero_len_arcs": 106,
  "duplicate_arcs": 1903,
  "isolated_nodes": 0,
  "arcs_bad_node_ref": 0,
  "trips_multi_weak_components": 89,
  "trips_multi_strong_components": 90,
  "min_largest_weak_frac": 0.6927966101694916,
  "arc_len_max": 9307.844613577989,
  "dt_gt1": 7550,
  "dt_gt5": 1260,
  "dt_gt30": 135,
  "dt_max": 153064.0,
  "n_invalid_coord": 0,
  "n_out_of_order": 0,
  "n_dup_timestamp": 0,
  "n_dup_rows": 0,
  "n_raw_rows":

In [17]:
import sys, json, shutil, importlib
import numpy as np, pandas as pd
from pathlib import Path
sys.path.insert(0, '/tmp/pathsnap_p1/src')
import pathsnap; importlib.reload(pathsnap)
from pathsnap import *
from google.colab import drive
drive.mount('/content/drive')

raw_root = Path('/tmp/raw_data/Student folder'); OUT = Path('/content/pathsnap_out')
DEST = Path('/content/drive/MyDrive/ML HACKATHON'); DEST.mkdir(parents=True, exist_ok=True)
splits = json.load(open(OUT / 'splits.json')); S = json.load(open(OUT / 'audit_summary.json'))


old = DEST / 'old_v0'; old.mkdir(exist_ok=True)
for nm in ['candidates_train.parquet', 'candidates_student.parquet']:
    if (DEST / nm).exists() and not (old / nm).exists():
        shutil.move(str(DEST / nm), str(old / nm))

def free_gb(): return shutil.disk_usage('/content/drive')[2] / 1e9
def push(files, subdir=None):
    dst = DEST / subdir if subdir else DEST; dst.mkdir(exist_ok=True)
    need = sum(Path(f).stat().st_size for f in files) / 1e9
    assert need < free_gb() * 0.95, f"Drive needs {need:.2f} GB, only {free_gb():.2f} GB free - free some space and re-run this cell"
    for f in files: shutil.copy(f, dst / Path(f).name)
    print(f"  -> Drive/{dst.name}: {len(files)} files, {need*1e3:.0f} MB (Drive free {free_gb():.2f} GB)")

# ---- static docs written before delivery ----
OBS_DOC = [
 ('trip_id','str','Trip folder name'), ('obs_idx','int','0-based index of the GPS point after cleaning, time order'),
 ('orig_row','int','0-based row in the raw .track file'), ('timestamp','s','Raw timestamp'),
 ('dt_prev','s','Seconds since previous point (0 for first point and for duplicate timestamps)'),
 ('is_gap','bool',f'dt_prev > {GAP_S:.0f} s'), ('lon','deg','Longitude (WGS84)'), ('lat','deg','Latitude (WGS84)'),
 ('x','m','East, local projection, origin = trip lon0/lat0 (trips_meta)'), ('y','m','North, same projection'),
 ('speed','m/s','Step distance / max(dt_prev, 1 s)'),
 ('heading','deg','Compass bearing of movement from previous point (0 = N, clockwise); carried forward if moved < 1 m'),
 ('n_candidates','count','Candidates kept for this point (can be 0)'),
 ('true_arc_id','int','Reference arc from monotone alignment to .route (labelled data only)'),
 ('label_quality','str','ok / ambiguous / far / gap / unlabeled (labelled data only)'),
 ('heading_valid','bool','True if the point moved >= 1 m so its own heading is measured, not carried forward'),
 ('true_dist_m','m','Distance from the point to its true arc (labelled data only)')]
CAND_DOC = [
 ('trip_id','str','Trip folder name'), ('obs_idx','int','Join key to observations'),
 ('arc_id','int','Row index in that trip\'s .arcs (PER TRIP)'), ('from_node','int','Arc from-node id (per trip)'),
 ('to_node','int','Arc to-node id (per trip)'), ('dist_to_arc_m','m','Point-to-segment distance'),
 ('snap_x','m','Snapped point x on the arc (same projection)'), ('snap_y','m','Snapped point y'),
 ('offset_m','m','Distance along the arc from from_node to the snap point'), ('arc_length_m','m','Arc length'),
 ('bearing_diff_deg','deg 0-180','|heading - arc bearing|; arc bearing is from_node -> to_node'),
 ('cand_rank','int','0 = closest candidate for that point'), ('label','0/1','1 = this arc is the reference arc')]
pd.DataFrame([('observations',)+r for r in OBS_DOC] + [('candidates',)+r for r in CAND_DOC],
             columns=['table','column','unit','meaning']).to_csv(OUT/'columns_dictionary.csv', index=False)
json.dump({'schema_version': SCHEMA_VERSION, 'radius_m': RADIUS_M, 'max_candidates': MAX_CANDIDATES, 'gap_s': GAP_S,
           'observations_columns': OBS_COLS_LABELED, 'candidates_columns': CAND_COLS,
           'sort_order': 'candidates: trip_id, obs_idx, cand_rank; observations: trip_id, obs_idx',
           'ids': 'arc_id and node_id are PER TRIP; global key = (trip_id, arc_id)',
           'projection': PROJECTION, 'label_quality_values': ['ok', 'ambiguous', 'far', 'gap', 'unlabeled'],
           'thresholds': {'ambiguous_m': AMBIG_M, 'far_m': FAR_M, 'align_lambda_m_per_route_pos': ALIGN_LAMBDA}},
          open(OUT/'schema.json', 'w'), indent=2)
(OUT/'CHANGELOG.md').write_text(f"""# PathSnap data changelog
## {SCHEMA_VERSION} (2026-11-18) first frozen schema
- Replaces old_v0 candidates (19 features + _scaled columns).
- radius {RADIUS_M:.0f} m, max {MAX_CANDIDATES} candidates per point.
Policy: any change to columns, radius or max_candidates bumps the version here and is announced BEFORE files are replaced.
""")
shutil.copy('/tmp/pathsnap_p1/src/pathsnap.py', OUT / 'pathsnap.py')
DOCS = [OUT/'pathsnap.py', OUT/'splits.json', OUT/'schema.json', OUT/'columns_dictionary.csv', OUT/'CHANGELOG.md']

# ---- processing ----
def process(label, tids, outdir):
    outdir.mkdir(parents=True, exist_ok=True)
    O, C, A_, N_, R_, M, dts = [], [], [], [], [], [], []
    for tid in tids:
        d = load_trip(tid, raw_root=raw_root)
        obs, cand, info = build_observations_and_candidates(tid, d['track'], d['nodes'], d['arcs'])
        obs, cand, lab = attach_labels(obs, cand, d['nodes'], d['arcs'], d['route'], info)
        a, nd, rt = graph_tables(tid, d['nodes'], d['arcs'], d['route'], info['lat0'], info['lon0'])
        O.append(obs); C.append(cand); A_.append(a); N_.append(nd); R_.append(rt); dts.append(lab['d_true'])
        n = info['n_obs']; dt_ = lab['d_true']
        M.append({'trip_id': tid, 'lat0': info['lat0'], 'lon0': info['lon0'], **info['cleaning'], 'n_obs': n,
                  'n_zero_cand': info['n_zero_cand'], 'n_truncated': info['n_truncated'], 'n_labeled': lab['n_labeled'],
                  'n_hit': lab['n_hit'], 'n_miss': lab['n_miss'], 'n_miss_radius': lab['n_miss_radius'], 'n_miss_cap': lab['n_miss_cap'],
                  'miss_rate': lab['n_miss'] / max(lab['n_labeled'], 1),
                  'mean_dist_to_true_m': float(dt_.mean()) if len(dt_) else np.nan,
                  'p95_dist_to_true_m': float(np.percentile(dt_, 95)) if len(dt_) else np.nan,
                  **{f'q_{k}': v for k, v in lab['quality'].items()},
                  **{f'n_within_{r}m': v for r, v in lab['recall_counts'].items()}})
    obs = pd.concat(O, ignore_index=True); cand = pd.concat(C, ignore_index=True)
    for df, keys in ((obs, ['trip_id', 'obs_idx']), (cand, ['trip_id', 'obs_idx', 'cand_rank'])):
        k = df[keys].reset_index(drop=True)
        assert k.equals(k.sort_values(keys, kind='stable').reset_index(drop=True)), f"{label}: not sorted by {keys}"
    meta = pd.DataFrame(M)
    files = []
    for nm, df in (('observations', obs), ('candidates', cand), ('arcs', pd.concat(A_, ignore_index=True)),
                   ('nodes', pd.concat(N_, ignore_index=True)), ('routes', pd.concat(R_, ignore_index=True))):
        p = outdir / f'{nm}_{label}.parquet'; df.to_parquet(p, index=False, compression='zstd'); files.append(p)
    meta.to_csv(outdir / f'trips_meta_{label}.csv', index=False); files.append(outdir / f'trips_meta_{label}.csv')
    meta[['trip_id', 'n_obs', 'n_labeled', 'n_hit', 'n_miss', 'n_miss_radius', 'n_miss_cap', 'miss_rate']].to_csv(
        outdir / f'miss_rate_per_trip_{label}.csv', index=False); files.append(outdir / f'miss_rate_per_trip_{label}.csv')
    tot = max(int(meta.n_labeled.sum()), 1)
    rc = pd.DataFrame({'radius_m': list(RECALL_RADII), 'recall': [meta[f'n_within_{r}m'].sum() / tot for r in RECALL_RADII]})
    rc.loc[len(rc)] = [f'{RADIUS_M:.0f} (actual, cap {MAX_CANDIDATES})', 1 - meta.n_miss.sum() / tot]
    rc.to_csv(outdir / f'recall_vs_radius_{label}.csv', index=False); files.append(outdir / f'recall_vs_radius_{label}.csv')
    print(f"[{label}] obs={obs.shape} cand={cand.shape} trips={len(tids)} "
          f"overall miss={meta.n_miss.sum()/tot:.3%} zero-cand pts={int(meta.n_zero_cand.sum())}")
    return {'obs': obs, 'cand': cand, 'meta': meta, 'recall': rc, 'files': files,
            'd_true': np.concatenate(dts) if dts else np.empty(0)}

res = {}
first = (splits['student'] or splits['train'])[:1]
print("1/3 sample trip", first)
res['sample'] = process('sample', first, OUT / 'sample')
push(res['sample']['files'] + DOCS, 'sample'); print("   sample delivered - tell the team")

print("2/3 student split")
res['student'] = process('student', splits['student'], OUT / 'student')
push(res['student']['files']); print("   student delivered - tell the team")

print("3/3 train split")
res['train'] = process('train', splits['train'], OUT / 'train')
push(res['train']['files']); print("   train delivered")

# ---- report ----
def pct(a, b): return f"{100*a/max(b,1):.2f}%"
allm = pd.concat([res['student']['meta'], res['train']['meta']])
dtrue = np.concatenate([res['student']['d_true'], res['train']['d_true']])
qsum = {q: int(allm[f'q_{q}'].sum()) for q in ('ok', 'ambiguous', 'far', 'gap')}
nl = int(allm.n_labeled.sum())
shared = S['unique_nodes_files'] < S['n_trips']
rec_tbl = lambda s: res[s]['recall'].to_string(index=False)
report = f"""# PathSnap P1 data report (schema {SCHEMA_VERSION})
Trips: {S['trips_per_split']}. Search radius {RADIUS_M:.0f} m, max {MAX_CANDIDATES} candidates per point (closest first).

## Loader, splits, conventions
- `pathsnap.load_trip(trip_id, raw_root=...)` or `load_trip(trip_id, data_dir=...)` returns track, nodes, arcs, route. `splits.json` lists train / student / evaluator trip ids (evaluator list is empty if no such folder was in the zip).
- Order is (lon, lat) in every file. Verified by bounding box on {S['lonlat_ok_trips']}/{S['n_trips']} trips; a swapped reading failed (see audit_trips.csv). Typical coordinates: lon {S['median_lon']:.3f}, lat {S['median_lat']:.3f}.
- Projection: {PROJECTION} Helpers in pathsnap.py: project, unproject, haversine_m. Max relative error vs haversine on steps > 10 m: {S['proj_max_rel_err']:.2e}.
- IDs: arc_id = row index in that trip's .arcs, node_id = row index in .nodes. {'The nodes/arcs files are identical across some trips (' + str(S['unique_nodes_files']) + ' unique node files for ' + str(S['n_trips']) + ' trips), so check whether IDs are shared before relying on per-trip keys.' if shared else 'Every trip has its own graph (' + str(S['unique_nodes_files']) + ' unique node files for ' + str(S['n_trips']) + ' trips), so IDs are PER TRIP; the global key is (trip_id, arc_id).'}
- Direction: arcs are directed (from_node -> to_node). On average {100*S['frac_arcs_with_reverse_mean']:.1f}% of arcs have a reverse arc.

## Graph and route health
- Consecutive .route arcs: {S['route_connected']:,} connected of {S['route_pairs']:,} pairs ({pct(S['route_connected'], S['route_pairs'])}); repeated arc {S['route_repeat']:,}; U-turn {S['route_uturn']:,}; disconnected {S['route_disconnected']:,}; invalid ids {S['route_invalid_ids']}.
- Self-loops {S['self_loops']}, zero-length arcs {S['zero_len_arcs']}, duplicate arcs {S['duplicate_arcs']}, isolated nodes {S['isolated_nodes']}, bad node refs {S['arcs_bad_node_ref']}.
- Trips with more than one weakly connected component: {S['trips_multi_weak_components']}; with more than one strongly connected component: {S['trips_multi_strong_components']}. Longest arc {S['arc_len_max']:.0f} m (arcs > {LONG_ARC_M:.0f} m are checked against every point).

## GPS timing and cleaning
- Gaps > 1 s: {S['dt_gt1']:,}; > 5 s: {S['dt_gt5']:,}; > 30 s: {S['dt_gt30']:,}; largest gap {S['dt_max']:.0f} s (per trip in audit_trips.csv: dt_max).
- Cleaning on {S['n_raw_rows']:,} raw rows: invalid coordinates dropped {S['n_invalid_coord']}; out-of-order steps {S['n_out_of_order']} (stable sort by timestamp if any); duplicate timestamps {S['n_dup_timestamp']} (kept, dt_prev = 0); duplicate rows {S['n_dup_rows']} (kept). `orig_row` maps back to the raw file.

## Labels
- Per-point labels come from a monotone Viterbi alignment of the GPS points to the .route arcs (position never moves backwards; cost = point-to-arc distance + {ALIGN_LAMBDA} m per route position skipped forward). `true_arc_id` is the aligned arc; `label` = 1 for that arc in the candidates.
- label_quality: far = true arc > {FAR_M:.0f} m away; ambiguous = another route arc within {AMBIG_M:.0f} m of the assigned one; gap = point after a gap > {GAP_S:.0f} s; ok = none of these.
- Counts (student + train): ok {qsum['ok']:,} ({pct(qsum['ok'], nl)}), ambiguous {qsum['ambiguous']:,}, far {qsum['far']:,}, gap {qsum['gap']:,}.
- GPS to true arc distance: mean {dtrue.mean():.2f} m, 95th percentile {np.percentile(dtrue, 95):.2f} m.
- The raw .route is kept in routes_*.parquet (trip_id, route_pos, arc_id) for sequence-level metrics.

## Candidate coverage
- Miss rate = share of GPS points whose true arc is not among the candidates. Train {res['train']['meta'].n_miss.sum()/max(res['train']['meta'].n_labeled.sum(),1):.3%}, student {res['student']['meta'].n_miss.sum()/max(res['student']['meta'].n_labeled.sum(),1):.3%}. Per trip: miss_rate_per_trip_*.csv (split into radius misses and cap misses).
- Recall vs radius, train:
{rec_tbl('train')}
- Recall vs radius, student:
{rec_tbl('student')}

## Functions
- `build_observations_and_candidates(trip_id, track, nodes, arcs)` is label-free and API-safe.
- `attach_labels(obs, cand, nodes, arcs, route, info)` is for train/student data and server-side scoring only.
- Rows sorted by trip_id, obs_idx, cand_rank. Any change to schema or radius: see CHANGELOG.md, announced before files are replaced.
"""
(OUT / 'data_report.md').write_text(report); print(report)
push([OUT/'data_report.md', OUT/'audit_trips.csv', OUT/'audit_summary.json'])
print("\nDONE. Final Drive folder:")
for fp in sorted(DEST.rglob('*')):
    if fp.is_file(): print(f"{str(fp.relative_to(DEST)):55s} {fp.stat().st_size/1e6:8.1f} MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1/3 sample trip ['00000000']
[sample] obs=(2503, 17) cand=(17174, 13) trips=1 overall miss=0.040% zero-cand pts=0
  -> Drive/sample: 13 files, 1 MB (Drive free 1.98 GB)
   sample delivered - tell the team
2/3 student split
[student] obs=(19160, 17) cand=(246606, 13) trips=10 overall miss=0.120% zero-cand pts=0
  -> Drive/ML HACKATHON: 8 files, 15 MB (Drive free 1.98 GB)
   student delivered - tell the team
3/3 train split
[train] obs=(208417, 17) cand=(2168787, 13) trips=80 overall miss=10.922% zero-cand pts=13574
  -> Drive/ML HACKATHON: 8 files, 220 MB (Drive free 1.96 GB)
   train delivered
# PathSnap P1 data report (schema 1.0)
Trips: {'train': 80, 'student': 10}. Search radius 75 m, max 20 candidates per point (closest first).

## Loader, splits, conventions
- `pathsnap.load_trip(trip_id, raw_root=...)` or `load_trip(trip_id, data_dir=...)` returns track

In [18]:
from google.colab import drive, auth
from pathlib import Path


drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

DEST = Path('/content/drive/MyDrive/ML HACKATHON')
print("Files Colab sees:", sum(1 for f in DEST.rglob('*') if f.is_file()))


auth.authenticate_user()
from googleapiclient.discovery import build
svc = build('drive', 'v3')
folders = svc.files().list(
    q="name='ML HACKATHON' and mimeType='application/vnd.google-apps.folder' and trashed=false",
    fields="files(id,name,modifiedTime,owners(emailAddress),webViewLink)").execute()['files']
for fo in folders:
    kids = svc.files().list(q=f"'{fo['id']}' in parents and trashed=false",
                            fields="files(name)", pageSize=100).execute()['files']
    print("\nFOLDER:", fo['webViewLink'])
    print("  modified:", fo['modifiedTime'], "| owner:", fo['owners'][0]['emailAddress'])
    print("  items:", sorted(k['name'] for k in kids))

Mounted at /content/drive
Files Colab sees: 35



FOLDER: https://drive.google.com/drive/folders/1342ob9YBsZM3m34dyDkIlpmidG1kusfZ
  modified: 2026-11-18T06:17:22.892Z | owner: tarungowdap02@gmail.com
  items: ['arcs_student.parquet', 'arcs_train.parquet', 'audit_summary.json', 'audit_trips.csv', 'candidates_student.parquet', 'candidates_train.parquet', 'data_report.md', 'miss_rate_per_trip_student.csv', 'miss_rate_per_trip_train.csv', 'nodes_student.parquet', 'nodes_train.parquet', 'observations_student.parquet', 'observations_train.parquet', 'old_v0', 'preprocessor.joblib', 'recall_vs_radius_student.csv', 'recall_vs_radius_train.csv', 'routes_student.parquet', 'routes_train.parquet', 'sample', 'trips_meta_student.csv', 'trips_meta_train.csv']
